# 序列逆置 （加注意力的seq2seq）
使用attentive sequence to sequence 模型将一个字符串序列逆置。例如 `OIMESIQFIQ` 逆置成 `QIFQISEMIO`(下图来自网络，是一个加attentino的sequence to sequence 模型示意图)
![attentive seq2seq](./seq2seq-attn.jpg)

In [1]:
import numpy as np
import tensorflow as tf
import collections
from tensorflow import keras
from tensorflow.keras import layers
from tensorflow.keras import layers, optimizers, datasets
import os,sys,tqdm

## 玩具序列数据生成
生成只包含[A-Z]的字符串，并且将encoder输入以及decoder输入以及decoder输出准备好（转成index）

In [2]:
import random
import string

def randomString(stringLength):
    """Generate a random string with the combination of lowercase and uppercase letters """

    letters = string.ascii_uppercase
    return ''.join(random.choice(letters) for i in range(stringLength))

def get_batch(batch_size, length):
    batched_examples = [randomString(length) for i in range(batch_size)]
    enc_x = [[ord(ch)-ord('A')+1 for ch in list(exp)] for exp in batched_examples]
    y = [[o for o in reversed(e_idx)] for e_idx in enc_x]
    dec_x = [[0]+e_idx[:-1] for e_idx in y]
    return (batched_examples, tf.constant(enc_x, dtype=tf.int32), 
            tf.constant(dec_x, dtype=tf.int32), tf.constant(y, dtype=tf.int32))
print(get_batch(2, 10))

(['TGRTKRIQLH', 'MJBGRNEGTJ'], <tf.Tensor: shape=(2, 10), dtype=int32, numpy=
array([[20,  7, 18, 20, 11, 18,  9, 17, 12,  8],
       [13, 10,  2,  7, 18, 14,  5,  7, 20, 10]], dtype=int32)>, <tf.Tensor: shape=(2, 10), dtype=int32, numpy=
array([[ 0,  8, 12, 17,  9, 18, 11, 20, 18,  7],
       [ 0, 10, 20,  7,  5, 14, 18,  7,  2, 10]], dtype=int32)>, <tf.Tensor: shape=(2, 10), dtype=int32, numpy=
array([[ 8, 12, 17,  9, 18, 11, 20, 18,  7, 20],
       [10, 20,  7,  5, 14, 18,  7,  2, 10, 13]], dtype=int32)>)


# 建立sequence to sequence 模型

完成两空，模型搭建以及单步解码逻辑

In [12]:
class mySeq2SeqModel(keras.Model):
    def __init__(self):
        super(mySeq2SeqModel, self).__init__()
        self.v_sz = 27
        self.hidden = 128
        self.embed_layer = tf.keras.layers.Embedding(self.v_sz, 64)
        
        self.encoder_cell = tf.keras.layers.SimpleRNNCell(self.hidden)
        self.decoder_cell = tf.keras.layers.SimpleRNNCell(self.hidden)
        
        self.encoder = tf.keras.layers.RNN(self.encoder_cell, 
                                           return_sequences=True, return_state=True)
        self.decoder = tf.keras.layers.RNN(self.decoder_cell, 
                                           return_sequences=True, return_state=True)
        
        self.dense_attn = tf.keras.layers.Dense(1)
        self.input_proj = tf.keras.layers.Dense(self.hidden)
        self.dense = tf.keras.layers.Dense(self.v_sz)
        
    def call(self, enc_ids, dec_ids):
        enc_emb = self.embed_layer(enc_ids)
        enc_out, enc_state = self.encoder(enc_emb)
        
        dec_state = enc_state
        dec_emb = self.embed_layer(dec_ids)
        
        dec_len = tf.shape(dec_emb)[1]
        logits_list = []
        
        # 使用 tf.range 代替 Python range
        for t in tf.range(dec_len):
            dec_input = dec_emb[:, t, :]
            
            dec_state_exp = tf.expand_dims(dec_state, axis=1)
            score = self.dense_attn(tf.tanh(enc_out + dec_state_exp))
            score = tf.squeeze(score, axis=-1)
            attn_weights = tf.nn.softmax(score, axis=-1)
            context = tf.reduce_sum(enc_out * tf.expand_dims(attn_weights, axis=-1), axis=1)
            
            rnn_input = tf.concat([dec_input, context], axis=-1)
            cell_input = self.input_proj(rnn_input)
            
            out, dec_state = self.decoder_cell(cell_input, [dec_state])
            dec_state = dec_state[0]
            
            logits = self.dense(out)
            logits_list.append(logits)
        
        logits = tf.stack(logits_list, axis=1)
        return logits
    
    def encode(self, enc_ids):
        enc_emb = self.embed_layer(enc_ids)
        enc_out, enc_state = self.encoder(enc_emb)
        return enc_out, [enc_out[:, -1, :], enc_state]
    
    def get_next_token(self, x, state, enc_out):
        x_emb = self.embed_layer(tf.expand_dims(x, axis=-1))
        x_emb = tf.squeeze(x_emb, axis=1)
        
        dec_state = state[1]
        
        dec_state_exp = tf.expand_dims(dec_state, axis=1)
        score = self.dense_attn(tf.tanh(enc_out + dec_state_exp))
        score = tf.squeeze(score, axis=-1)
        attn_weights = tf.nn.softmax(score, axis=-1)
        context = tf.reduce_sum(enc_out * tf.expand_dims(attn_weights, axis=-1), axis=1)
        
        rnn_input = tf.concat([x_emb, context], axis=-1)
        cell_input = self.input_proj(rnn_input)
        
        out, new_state = self.decoder_cell(cell_input, [dec_state])
        new_state = new_state[0]
        
        logits = self.dense(out)
        next_token = tf.argmax(logits, axis=-1, output_type=tf.int32)
        
        return next_token, [out, new_state]

# Loss函数以及训练逻辑

In [13]:
@tf.function
def compute_loss(logits, labels):
    losses = tf.nn.sparse_softmax_cross_entropy_with_logits(
            logits=logits, labels=labels)
    losses = tf.reduce_mean(losses)
    return losses

#@tf.function
def train_one_step(model, optimizer, enc_x, dec_x, y):
    with tf.GradientTape() as tape:
        logits = model(enc_x, dec_x)
        loss = compute_loss(logits, y)

    # compute gradient
    grads = tape.gradient(loss, model.trainable_variables)
    optimizer.apply_gradients(zip(grads, model.trainable_variables))
    return loss

def train(model, optimizer, seqlen):
    loss = 0.0
    accuracy = 0.0
    for step in range(2000):
        batched_examples, enc_x, dec_x, y = get_batch(32, seqlen)
        loss = train_one_step(model, optimizer, enc_x, dec_x, y)
        if step % 500 == 0:
            print('step', step, ': loss', loss.numpy())
    return loss

# 训练迭代

In [14]:
optimizer = optimizers.Adam(0.0005)
model = mySeq2SeqModel()
train(model, optimizer, seqlen=20)

step 0 : loss 3.3094964
step 500 : loss 1.4029067
step 1000 : loss 0.83727896
step 1500 : loss 0.60980225


<tf.Tensor: shape=(), dtype=float32, numpy=0.390857994556427>

# 测试模型逆置能力
首先要先对输入的一个字符串进行encode，然后在用decoder解码出逆置的字符串

测试阶段跟训练阶段的区别在于，在训练的时候decoder的输入是给定的，而在预测的时候我们需要一步步生成下一步的decoder的输入

In [15]:
def sequence_reversal():
    def decode(init_state, steps, enc_out):
        b_sz = tf.shape(init_state[0])[0]
        cur_token = tf.zeros(shape=[b_sz], dtype=tf.int32)
        state = init_state
        collect = []
        for i in range(steps):
            cur_token, state = model.get_next_token(cur_token, state, enc_out)
            collect.append(tf.expand_dims(cur_token, axis=-1))
        out = tf.concat(collect, axis=-1).numpy()
        out = [''.join([chr(idx+ord('A')-1) for idx in exp]) for exp in out]
        return out
    
    batched_examples, enc_x, _, _ = get_batch(32, 20)
    enc_out, state = model.encode(enc_x)
    return decode(state, enc_x.get_shape()[-1], enc_out), batched_examples

def is_reverse(seq, rev_seq):
    rev_seq_rev = ''.join([i for i in reversed(list(rev_seq))])
    if seq == rev_seq_rev:
        return True
    else:
        return False
print([is_reverse(*item) for item in list(zip(*sequence_reversal()))])
print(list(zip(*sequence_reversal())))

[True, True, False, False, False, False, False, False, False, False, False, False, False, False, False, False, False, False, False, True, False, False, False, True, True, False, True, False, False, False, False, False]
[('BCSMUZSYFLUPNYBSOILQ', 'ZIOWNQUEBYPUFSZYMSCB'), ('PPEQJWCMZZIQQBGXEFAW', 'XABEBKTPCIZZMCWJQEPP'), ('ALGUIUTSDKKRTRDAGLCU', 'BDQGTIBFMMRDKTUIUGLA'), ('EKDKWCBFAUAAXYAKMYNS', 'JNGMKTSXAAUAFBCWKDKE'), ('RYXBXZNEHQPWYUAHIFML', 'UPYYHKRANYQHGNZXBXYR'), ('WUBQHQRPFBDFJGAEDMZR', 'RZMDEAGJFDBFPRQHQBUW'), ('RWHPTBYLZDGDXTAYUWMA', 'AWRUYATXDGDZLYBTPHWR'), ('DJMSGJLFIZRXYCLSSXBD', 'WVVSSXZIXRZIFLJGSMJD'), ('LCUVAVIUPQENVLILKSLR', 'XLKRBILQJEQPUIVAVUCL'), ('FNMUKGAVJWNKEKTQRGVY', 'KTGWQGWKIAJCKAAGUMJF'), ('VAJUHHBSZLSOSDTKULKZ', 'CKZYSOUKQSBSEUHJFMAV'), ('TOHBIGCUUAKJWLJATVXT', 'BXATAJLWJKAUUCGIBHOT'), ('DMIYPTAOQQYQMTVPIJVI', 'CVJILPTMQYQQOATPYIMD'), ('OAUMGVLCCVEKXNJYGWMF', 'AWRGYJNXKEVCCLVGMUAO'), ('GJQQDJFCAKQFYHAEOIFZ', 'ZZSREHDZJVBACFJDQQJG'), ('RURBTTTBOAXFNMSFQOUA', 'BCAQ